In [3]:
import geopandas as gpd
import pandas as pd
from shapely.geometry import Polygon
import numpy as np
import matplotlib.pyplot as plt

In [4]:
def load_neighborhoods(filter_by_bbox = False):
    filename = './data/Barrios_2024_CLBB'
    neighborhoods = gpd.read_file(filename)
    neighborhoods.drop(['Id'], axis=1, inplace=True)
    neighborhoods.reset_index(inplace=True)
    neighborhoods.rename(columns={'index': 'nb_id'}, inplace=True)
    neighborhoods.to_crs('EPSG:4326', inplace=True)

    if filter_by_bbox:
        bbox = {
            'lat_min': -36.71672579573574,
            'lat_max': -36.70690973726542,
            'lon_min': -73.11671311007053,
            'lon_max': -73.10692144785894,
        }

        # create a polygon from the bounding box
        bbox_polygon = Polygon([
            (bbox['lon_min'], bbox['lat_min']),
            (bbox['lon_min'], bbox['lat_max']),
            (bbox['lon_max'], bbox['lat_max']),
            (bbox['lon_max'], bbox['lat_min'])
        ])

        bbox_polygon = gpd.GeoDataFrame(index=[0], crs='EPSG:4326', geometry=[bbox_polygon])

        neighborhoods = gpd.overlay(neighborhoods, bbox_polygon, how='intersection')

    return neighborhoods

In [5]:
default_crs = "EPSG:4326"
scenario = 'Actual'

def load_land_uses(scenario: str, filter_by_bbox = False):
    landuses = gpd.read_file(f'./data/usos_de_suelo/{scenario}')
    try:
        landuses = landuses.to_crs('EPSG:4326')
    except Exception as e:
        print(e)
        pass

    if filter_by_bbox:
        bbox = {
            'lat_min': -36.71672579573574,
            'lat_max': -36.70690973726542,
            'lon_min': -73.11671311007053,
            'lon_max': -73.10692144785894,
        }

        # create a polygon from the bounding box
        bbox_polygon = Polygon([
            (bbox['lon_min'], bbox['lat_min']),
            (bbox['lon_min'], bbox['lat_max']),
            (bbox['lon_max'], bbox['lat_max']),
            (bbox['lon_max'], bbox['lat_min'])
        ])

        bbox_polygon = gpd.GeoDataFrame(index=[0], crs='EPSG:4326', geometry=[bbox_polygon])
    
        landuses = gpd.overlay(landuses, bbox_polygon, how='intersection')
    landuses.reset_index(inplace=True)
    landuses.rename(columns={'index': 'lu_id'}, inplace=True)
    return landuses

In [6]:
def calc_diversity_in_neighborhoods(landuses: gpd.GeoDataFrame, neighborhoods: gpd.GeoDataFrame):
    # add the tag to each landuses polygon to what neighborhood it belongs to
    # use geopandas functions as much as possible like sjoin
    landuses_neighborhoods = gpd.overlay(landuses, neighborhoods)
    landuses_neighborhoods.to_crs('EPSG:32718', inplace=True)
    landuses_neighborhoods['area'] = landuses_neighborhoods.area
    landuses_neighborhoods.to_crs('EPSG:4326', inplace=True)

    group_cols = ['lu_id', 'nb_id']
    neighborhoods_index_on_land_uses = landuses_neighborhoods.groupby(group_cols)['area'].agg(['max']).reset_index()
    neighborhoods_index_on_land_uses.drop(['max'], axis=1, inplace=True)
    landuses_neighborhoods = pd.merge(landuses, neighborhoods_index_on_land_uses, on=['lu_id'])
    landuses_neighborhoods = pd.merge(landuses_neighborhoods, neighborhoods.drop(columns=['geometry', 'Area']), on=['nb_id'])

    landuses_neighborhoods.to_crs('EPSG:32718', inplace=True)
    landuses_neighborhoods['area'] = landuses_neighborhoods.area
    landuses_neighborhoods.to_crs('EPSG:4326', inplace=True)

    group_id_col = 'nb_id'
    gdf_group_by_use = landuses_neighborhoods.groupby([group_id_col, 'DESTINO']).agg({'area':'sum'}).reset_index().rename(columns={'area': 'area_by_use'})
    gdf_group_by_nb = landuses_neighborhoods.groupby([group_id_col]).agg({'area':'sum'}).reset_index().rename(columns={'area': 'area_used_by_nb'})

    gdf_group = pd.merge(gdf_group_by_nb, gdf_group_by_use, on=group_id_col, how='left')

    gdf_group['fraction_by_use'] = gdf_group['area_by_use'] / gdf_group['area_used_by_nb']
    gdf_group['info_by_use'] = -1*gdf_group['fraction_by_use']*np.log2(gdf_group['fraction_by_use'])

    gdf_group.loc[gdf_group['fraction_by_use']==1, 'info_by_use'] = 0
    gdf_group = gdf_group[[group_id_col, 'info_by_use']].groupby(group_id_col).agg({'info_by_use':'sum'}).reset_index().rename(columns={'info_by_use': 'diversity'})

    gdf_diversity = pd.merge(gdf_group, neighborhoods, on=group_id_col)
    # gdf_diversity.drop(columns=['area_hex'], inplace=True)
    gdf_diversity = gpd.GeoDataFrame(gdf_diversity, geometry='geometry')
    # gdf_diversity.set_crs('EPSG:4326', inplace=True)
    return gdf_diversity

In [7]:
scenarios = ['Actual', 'Futuro']
filter_by_bbox = True
output = {}
landuses_hist = {}
for scenario in scenarios:
    # scenario = scenarios[1]
    landuses = load_land_uses(scenario, filter_by_bbox)
    neighborhoods = load_neighborhoods(filter_by_bbox)
    
    landuses['scenario'] = scenario
    
    try:
        landuses_hist.append(landuses)
    except Exception as e:
        landuses_hist[scenario] = landuses
        pass
    
    output[scenario] = calc_diversity_in_neighborhoods(landuses, neighborhoods)
    output[scenario]['scenario'] = scenario

In [18]:
import os
indicator_name = 'land_uses_diversity_by_neighborhood'
value_col = 'diversity'
index_col = 'nb_id'
format = 'parquet'
scenarios_available = []

save_path = f'./export/'
export_folder = os.path.join(save_path, indicator_name)
os.makedirs(export_folder, exist_ok=True)

for scenario in scenarios:
    scenario_folder = os.path.join(export_folder, scenario)
    os.makedirs(scenario_folder, exist_ok=True)
    try:
        if format == 'parquet':
            export_filename = os.path.join(scenario_folder, f'{indicator_name}.parquet')
            output[scenario].to_parquet(export_filename, index=False)
        elif format == 'shp':
            if os.path.exists(scenario_folder):
                import shutil
                shutil.rmtree(scenario_folder)
            export_filename = os.path.join(scenario_folder, f'{indicator_name}')
            output[scenario].to_file(export_filename)
        scenarios_available.append(scenario)
    except Exception as e:
        print(e)
        pass

indicator_details = {
    'indicator_name': indicator_name,
    'value_col': value_col,
    'index_col': index_col,
    'format': format,
    'scenarios_available': scenarios_available
}
# Save indicator details as .json in the export folder with name same as export_name
import json
details_filename = os.path.join(export_folder, f'indicator_details.json')
with open(details_filename, 'w') as f:
    json.dump(indicator_details, f)

In [ ]:
# read details file from export/[indicator]

In [8]:
gdf_output = pd.concat(output)

In [9]:
gdf_output.reset_index(inplace=True)

In [49]:
gdf_output = gdf_output.pivot_table(index='nb_id', columns='scenario', values='diversity').reset_index()

In [54]:
gdf_output['diff'] = gdf_output['Futuro'] - gdf_output['Actual']

In [56]:
pd.merge(gdf_output, neighborhoods, on='nb_id')

,nb_id,Actual,Futuro,diff,Nombre,Area,geometry
0,94,2.455453,2.431318,-0.024136,David Fuentes,443937.774391,"POLYGON ((-73.11649 -36.71437, -73.11578 -36.7..."
1,95,1.408478,1.408478,0.000000,Cerro Buena Vista,250574.974662,"POLYGON ((-73.11637 -36.713, -73.11671 -36.713..."
2,98,0.805200,0.805200,0.000000,Vista Hermosa,157576.312140,"POLYGON ((-73.11491 -36.70714, -73.11473 -36.7..."
3,99,1.799449,1.898474,0.099024,Talcahuano Centro,103490.220389,"POLYGON ((-73.11494 -36.711, -73.11449 -36.711..."
4,166,1.529981,1.529981,0.000000,Cerro Cornou,110289.069158,"POLYGON ((-73.1159 -36.70841, -73.11481 -36.70..."
5,268,2.099910,3.292966,1.193057,Costanera Talcahuano,380575.463014,"POLYGON ((-73.11023 -36.71547, -73.11106 -36.7..."
